In [ ]:
import os
import json
import time
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from torch import nn
from torch.optim import AdamW

from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

#  Configuration

MODEL_NAME = "distilbert-base-uncased"

SEEDS = [1] #[1,42,999,1234]

NUM_TRAJ = 10
MAX_STEPS_PER_TRAJ = 15
NUM_EPOCHS = 3
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")



#  Dataset Preparation

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        idx = int(idx)

        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def load_emotion_dataset(model_name=MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    dataset = load_dataset("emotion")

    train_texts = list(dataset["train"]["text"])
    train_labels = list(dataset["train"]["label"])

    test_texts = list(dataset["validation"]["text"])
    test_labels = list(dataset["validation"]["label"])

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_texts,
        train_labels,
        test_size=0.2,
        random_state=42,
        stratify=train_labels
    )

    train_ds = EmotionDataset(train_texts, train_labels, tokenizer)
    val_ds   = EmotionDataset(val_texts,   val_labels,   tokenizer)
    test_ds  = EmotionDataset(test_texts,  test_labels,  tokenizer)

    return train_ds, val_ds, test_ds, tokenizer


# Utility helpers — bounds & normalization

def normalise_step(step_vec: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(step_vec)
    if norm == 0:
        return step_vec
    return step_vec / norm


def project_bounds(x: np.ndarray) -> np.ndarray:
    return np.clip(x, 0.0, 1.0)


#  BSO SEARCH — PART 2
#  LoRA model builder, decode(), fitness function, GPU logging


def decode_candidate(x):

    # learning_rate: log-uniform 1e-6 → 5e-4
    lr_min = 1e-6
    lr_max = 5e-4
    log_lr = np.log(lr_min) + x[0] * (np.log(lr_max) - np.log(lr_min))
    learning_rate = float(np.exp(log_lr))

    # warmup_ratio: 0.0 → 0.2
    warmup_ratio = float(x[1] * 0.2)

    # discrete values
    lora_r_vals = [2, 4, 8, 16, 32]
    lora_alpha_vals = [8, 16, 32, 64, 128]

    def pick(arr, v):
        idx = int(round(v * (len(arr) - 1)))
        return arr[max(0, min(len(arr) - 1, idx))]

    lora_r = pick(lora_r_vals, x[2])
    lora_alpha = pick(lora_alpha_vals, x[3])

    # dropout: 0.0 → 0.3
    lora_dropout = float(x[4] * 0.3)

    # target modules
    if x[5] < 0.5:
        target_modules = ["q_lin", "v_lin"]
        target_modules_option = "attention-only"
    else:
        target_modules = ["q_lin", "v_lin", "ffn.lin1", "ffn.lin2"]
        target_modules_option = "attention-plus-ffn"

    return {
        "learning_rate": learning_rate,
        "warmup_ratio": warmup_ratio,
        "lora_r": lora_r,
        "lora_alpha": lora_alpha,
        "lora_dropout": lora_dropout,
        "target_modules": target_modules,
        "target_modules_option": target_modules_option
    }



#  LoRA Model Trainer

def build_lora_model(config, num_labels=6):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels
    )

    peft_config = LoraConfig(
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        task_type="SEQ_CLS",
        target_modules=config["target_modules"]
    )

    model = get_peft_model(base_model, peft_config)
    return model


def train_and_eval_lora(config, train_ds, val_ds, tokenizer, device):
    model = build_lora_model(config).to(device)
    optimizer = AdamW(model.parameters(), lr=config["learning_rate"])

    total_steps = (len(train_ds) // BATCH_SIZE) * NUM_EPOCHS
    warmup_steps = int(config["warmup_ratio"] * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds,   batch_size=BATCH_SIZE)

    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            for key in batch:
                batch[key] = batch[key].to(device)

            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"    Epoch {epoch+1}/{NUM_EPOCHS} - train loss: {avg_loss:.4f}")

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            for key in batch:
                batch[key] = batch[key].to(device)

            outputs = model(**batch)
            preds = outputs.logits.argmax(dim=1)
            correct += (preds == batch["labels"]).sum().item()
            total += len(batch["labels"])

    return correct / total if total > 0 else 0.0

#  Fitness Function Wrapper

def fitness_function_wrapper(train_ds, val_ds, tokenizer, device):
    def fitness(hparams):
        return train_and_eval_lora(hparams, train_ds, val_ds, tokenizer, device)

    return fitness


#  BSO SEARCH — PART 3
#  Full Blindfolded Spiderman Optimisation Implementation

def bso_for_seed(
    fitness_function,
    seed: int,
    device: torch.device,
    dim: int = 6,
    num_trajectories: int = NUM_TRAJ,
    max_steps_per_trajectory: int = MAX_STEPS_PER_TRAJ,
):
    print(f"\n==============================")
    print(f" BSO for seed {seed} ")
    print(f"==============================")

    rng = np.random.default_rng(seed)

    theta = rng.uniform(0, np.pi/2)
    phi   = rng.uniform(0, 2*np.pi)
    x = rng.random(dim)

    cfg_x = decode_candidate(x)
    f_x = fitness_function(cfg_x)

    global_best_x = x.copy()
    global_best_val = f_x
    global_best_cfg = cfg_x

    all_val_accs = [f_x]
    gpu_mems = []
    total_train_time = 0.0
    trajectories_log = []

    for traj in range(1, num_trajectories + 1):
        print(f"\n=== Trajectory {traj}/{num_trajectories} ===")

        base_dir = np.array([
            np.sin(theta) * np.cos(phi),
            np.sin(theta) * np.sin(phi),
            np.cos(theta)
        ])

        direction = rng.normal(size=dim)
        direction = normalise_step(direction) * np.linalg.norm(base_dir)

        step_size = 1.0
        step_log = []
        best_val_traj = f_x

        for step_no in range(1, max_steps_per_trajectory + 1):

            y = x + step_size * direction
            y = project_bounds(y)

            if device.type == "cuda":
                torch.cuda.reset_peak_memory_stats()

            t0 = time.perf_counter()
            f_y = fitness_function(decode_candidate(y))
            t1 = time.perf_counter()

            eval_time = t1 - t0
            total_train_time += eval_time

            mem_gb = (
                torch.cuda.max_memory_allocated() / (1024**3)
                if device.type == "cuda" else None
            )
            if mem_gb is not None:
                gpu_mems.append(mem_gb)

            all_val_accs.append(f_y)

            step_log.append({
                "step_no": step_no,
                "move_type": "y-step",
                "val_acc": float(f_y),
                "train_time_sec": float(eval_time),
                "gpu_memory_gb": float(mem_gb) if mem_gb else None,
                "hyperparams": decode_candidate(y)
            })

            print(
                f"[BSO] Seed {seed} | Traj {traj} | Step {step_no} | "
                f"y-step={f_y:.4f} | best={global_best_val:.4f}"
            )

            if f_y > f_x:
                x = y.copy()
                f_x = f_y
                best_val_traj = max(best_val_traj, f_y)

                if f_y > global_best_val:
                    global_best_val = f_y
                    global_best_x = y.copy()
                    global_best_cfg = decode_candidate(y)

            else:
                dtheta = rng.uniform(-0.1, 0.1)
                dphi   = rng.uniform(-0.2, 0.2)

                theta_u = max(0.0, min(np.pi/2, theta + dtheta))
                phi_u   = (phi + dphi) % (2*np.pi)

                base_u = np.array([
                    np.sin(theta_u) * np.cos(phi_u),
                    np.sin(theta_u) * np.sin(phi_u),
                    np.cos(theta_u)
                ])

                u_step = rng.normal(size=dim)
                u_step = normalise_step(u_step) * np.linalg.norm(base_u)

                y_u = x + (step_size * 0.3) * u_step
                y_u = project_bounds(y_u)

                if device.type == "cuda":
                    torch.cuda.reset_peak_memory_stats()

                t0 = time.perf_counter()
                f_u = fitness_function(decode_candidate(y_u))
                t1 = time.perf_counter()

                eval_time_u = t1 - t0
                total_train_time += eval_time_u

                mem_u = (
                    torch.cuda.max_memory_allocated() / (1024**3)
                    if device.type == "cuda" else None
                )
                if mem_u is not None:
                    gpu_mems.append(mem_u)

                all_val_accs.append(f_u)

                step_log.append({
                    "step_no": step_no,
                    "move_type": "u-step",
                    "val_acc": float(f_u),
                    "train_time_sec": float(eval_time_u),
                    "gpu_memory_gb": float(mem_u) if mem_u else None,
                    "hyperparams": decode_candidate(y_u)
                })

                print(
                    f"[BSO] Seed {seed} | Traj {traj} | Step {step_no} | "
                    f"u-step={f_u:.4f} | best={global_best_val:.4f}"
                )

                if f_u > f_x:
                    x = y_u.copy()
                    f_x = f_u
                    best_val_traj = max(best_val_traj, f_u)

                    if f_u > global_best_val:
                        global_best_val = f_u
                        global_best_x = y_u.copy()
                        global_best_cfg = decode_candidate(y_u)
                else:
                    break

            theta = max(0.0, min(np.pi/2, theta + rng.uniform(-0.05, 0.05)))
            phi   = (phi + rng.uniform(-0.1, 0.1)) % (2*np.pi)

            step_size *= 0.85

        trajectories_log.append({
            "trajectory_no": traj,
            "steps": step_log,
            "best_val_acc_in_trajectory": float(best_val_traj)
        })

    arr = np.array(all_val_accs)
    val_acc_mean = float(arr.mean())
    val_acc_std = float(arr.std(ddof=0))

    avg_gpu = float(np.mean(gpu_mems)) if len(gpu_mems) > 0 else None

    best_traj_no = None
    best_traj_val = -np.inf
    for t in trajectories_log:
        if t["best_val_acc_in_trajectory"] > best_traj_val:
            best_traj_val = t["best_val_acc_in_trajectory"]
            best_traj_no = t["trajectory_no"]

    return {
        "seed": seed,
        "trajectories": trajectories_log,
        "best_trajectory": best_traj_no,
        "best_val_acc": float(global_best_val),
        "best_cfg": global_best_cfg,
        "val_acc_mean": val_acc_mean,
        "val_acc_std": val_acc_std,
        "all_val_accs": all_val_accs,
        "avg_train_time_sec": float(
            np.mean([s["train_time_sec"] for t in trajectories_log for s in t["steps"]])
        ),
        "total_train_time_sec": float(total_train_time),
        "avg_gpu_memory_gb": avg_gpu
    }


In [ ]:
#  BSO SEARCH — PART 4
#  Main loop, multi-seed execution, JSON creation and output

def main():
    print("\nLoading dataset...")
    train_ds, val_ds, test_ds, tokenizer = load_emotion_dataset(MODEL_NAME)

    print(f"Using device: {DEVICE}")

    # Build fitness wrapper (provides train_and_eval_lora)
    fitness_fn = fitness_function_wrapper(train_ds, val_ds, tokenizer, DEVICE)

    seeds_results = []
    global_best_val = -np.inf
    global_best_cfg = None
    global_best_seed = None
    global_best_trajectory = None

    total_train_time_all_seeds = 0.0


    #  Run BSO for each seed
    for seed in SEEDS:
        result = bso_for_seed(
            fitness_function=fitness_fn,
            seed=seed,
            device=DEVICE,
            dim=6,
            num_trajectories=NUM_TRAJ,
            max_steps_per_trajectory=MAX_STEPS_PER_TRAJ
        )

        seeds_results.append(result)
        total_train_time_all_seeds += result["total_train_time_sec"]

        # Track global best
        if result["best_val_acc"] > global_best_val:
            global_best_val = result["best_val_acc"]
            global_best_cfg = result["best_cfg"]
            global_best_seed = result["seed"]
            global_best_trajectory = result["best_trajectory"]


    # Mean & Std of seed best accuracies
    seed_best_vals = [s["best_val_acc"] for s in seeds_results]

    val_acc_mean_of_seed_bests = float(np.mean(seed_best_vals))
    val_acc_std_of_seed_bests  = float(np.std(seed_best_vals, ddof=0))

    #  Compute eval-level stats across ALL seeds
    all_evals = []
    for r in seeds_results:
        all_evals.extend(r["all_val_accs"])

    all_evals = np.array(all_evals)
    val_acc_mean_over_all_evals = float(all_evals.mean())
    val_acc_std_over_all_evals  = float(all_evals.std(ddof=0))


    search_space_dict = {
        "learning_rate": {
            "type": "log_uniform",
            "min": 1e-6,
            "max": 5e-4
        },
        "warmup_ratio": {
            "type": "continuous",
            "min": 0.0,
            "max": 0.2
        },
        "lora_r": {
            "type": "discrete",
            "values": [2, 4, 8, 16, 32]
        },
        "lora_alpha": {
            "type": "discrete",
            "values": [8, 16, 32, 64, 128]
        },
        "lora_dropout": {
            "type": "continuous",
            "min": 0.0,
            "max": 0.3
        },
        "target_modules": {
            "type": "categorical",
            "values": [
                "attention-only",
                "attention-plus-ffn"
            ]
        }
    }


    #  Construct final JSON

    output = {
        "method": "blindfolded_spiderman_optimisation",
        "gpu_name": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
        "num_seeds": len(SEEDS),
        "seeds": SEEDS,
        "num_trajectories_per_seed": NUM_TRAJ,
        "steps_per_trajectory": MAX_STEPS_PER_TRAJ,
        "num_epochs_per_eval": NUM_EPOCHS,
        "search_space": search_space_dict,

        "seeds_results": seeds_results,

        "overall_stats": {
            "global_best_seed": global_best_seed,
            "global_best_trajectory": global_best_trajectory,
            "global_best_val_acc": global_best_val,
            "global_best_hyperparams": global_best_cfg,

            "val_acc_mean_over_all_evals": val_acc_mean_over_all_evals,
            "val_acc_std_over_all_evals":  val_acc_std_over_all_evals,
            "total_train_time_sec_over_all_seeds": total_train_time_all_seeds,

            "val_acc_mean_of_seed_bests": val_acc_mean_of_seed_bests,
            "val_acc_std_of_seed_bests":  val_acc_std_of_seed_bests
        }
    }


    # Save JSON file
    os.makedirs("bso_results", exist_ok=True)
    out_path = f"bso_results/bso_output.json"

    with open(out_path, "w") as f:
        json.dump(output, f, indent=4)

    print(f"\nBSO results saved to: {out_path}")


#  Entry point
if __name__ == "__main__":
    main()



Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

[BSO] Seed 1 | Traj 9 | Step 1 | u-step=0.9281 | best=0.9350

=== Trajectory 10/10 ===
    Epoch 1/3 - train loss: 0.5588
    Epoch 2/3 - train loss: 0.1690
    Epoch 3/3 - train loss: 0.1061


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[BSO] Seed 1 | Traj 10 | Step 1 | y-step=0.9381 | best=0.9350
    Epoch 1/3 - train loss: 0.5466
    Epoch 2/3 - train loss: 0.1805
    Epoch 3/3 - train loss: 0.1146


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[BSO] Seed 1 | Traj 10 | Step 2 | y-step=0.9316 | best=0.9381
    Epoch 1/3 - train loss: 0.6093
    Epoch 2/3 - train loss: 0.2161
    Epoch 3/3 - train loss: 0.1210
[BSO] Seed 1 | Traj 10 | Step 2 | u-step=0.9328 | best=0.9381

BSO results saved to: bso_results/bso_output.json
